# GemVision — Model Evaluation (Colab)

Loads the **already-trained** gemstone CNN and evaluates it against the held-out test set, producing exactly the evidence a supervisor asks for: overall accuracy, a full precision/recall/F1 classification report, and confusion matrices.

No GPU needed for this (evaluation only, no training) — CPU runtime is fine, though a GPU will finish it faster.

**You'll need two things from your own machine, zipped up first:**
1. `backend/models/` (contains `gemstone_cnn.keras` and `class_indices.json`) — zip this folder.
2. `ml/data/gemstones-images/test/` (just the test folder, not train — much smaller) — zip this folder.

**Steps:** Run all cells top to bottom; the first upload cell asks for the model zip, the second for the test-data zip.

In [ ]:
from google.colab import files
import zipfile, os

print('Select your zipped backend/models/ folder (contains gemstone_cnn.keras + class_indices.json)...')
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name) as zf:
    zf.extractall('models_upload')

# Find the two files wherever they landed in the zip (handles both a flat
# zip and one that preserves the backend/models/ folder structure).
MODEL_PATH = None
CLASS_INDICES_PATH = None
for root, _, filenames in os.walk('models_upload'):
    for fn in filenames:
        if fn == 'gemstone_cnn.keras':
            MODEL_PATH = os.path.join(root, fn)
        elif fn == 'class_indices.json':
            CLASS_INDICES_PATH = os.path.join(root, fn)

assert MODEL_PATH, 'gemstone_cnn.keras not found in the uploaded zip'
assert CLASS_INDICES_PATH, 'class_indices.json not found in the uploaded zip'
print('Model:', MODEL_PATH)
print('Class indices:', CLASS_INDICES_PATH)

In [ ]:
print('Select your zipped ml/data/gemstones-images/test/ folder...')
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name) as zf:
    zf.extractall('test_data')

# Find the folder that actually contains the 87 class subfolders, whether
# the zip put them at the top level or nested one level down as "test/".
TEST_DIR = None
for root, dirs, _ in os.walk('test_data'):
    if len(dirs) >= 50:  # the 87 class folders
        TEST_DIR = root
        break
assert TEST_DIR, 'Could not find the class folders inside the uploaded zip'
print('Test data:', TEST_DIR, '-', len(os.listdir(TEST_DIR)), 'classes')

In [ ]:
import json
import numpy as np
import tensorflow as tf

IMAGE_SIZE = 224

with open(CLASS_INDICES_PATH) as f:
    class_indices = json.load(f)
class_names = [None] * len(class_indices)
for name, idx in class_indices.items():
    class_names[idx] = name

model = tf.keras.models.load_model(MODEL_PATH)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR, image_size=(IMAGE_SIZE, IMAGE_SIZE), batch_size=32,
    label_mode='categorical', shuffle=False,
)
# Sanity check: class_indices.json must describe the same class order the
# folder gives us, or every metric below would be silently wrong.
assert test_ds.class_names == class_names, 'Class order mismatch between class_indices.json and the test folder!'

y_true, y_pred = [], []
for batch_x, batch_y in test_ds:
    preds = model.predict(batch_x, verbose=0)
    y_true.extend(np.argmax(batch_y.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))
y_true, y_pred = np.array(y_true), np.array(y_pred)

accuracy = (y_true == y_pred).mean()
print(f'Overall test accuracy: {accuracy:.4f}  ({(y_true==y_pred).sum()}/{len(y_true)})')

In [ ]:
from sklearn.metrics import classification_report

# Per-class precision/recall/F1 -- this table is the metrics evidence.
report = classification_report(y_true, y_pred, target_names=class_names, zero_division=0)
print(report)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true, y_pred, labels=range(len(class_names)))

# Readable subset: the 20 classes with the most misclassifications, either
# as a source of errors or a magnet for other classes' errors.
off_diagonal = cm.copy()
np.fill_diagonal(off_diagonal, 0)
confusion_score = off_diagonal.sum(axis=0) + off_diagonal.sum(axis=1)
top20_idx = sorted(np.argsort(confusion_score)[::-1][:20])
cm_top20 = cm[np.ix_(top20_idx, top20_idx)]
top20_names = [class_names[i] for i in top20_idx]

fig, ax = plt.subplots(figsize=(12, 12))
ConfusionMatrixDisplay(cm_top20, display_labels=top20_names).plot(
    ax=ax, xticks_rotation=90, colorbar=True, cmap='Purples', values_format='d'
)
ax.set_title(f'GemVision CNN — 20 Most-Confused Classes (test accuracy {accuracy:.1%})')
fig.tight_layout()
fig.savefig('cnn_confusion_matrix_top20.png', dpi=150)
plt.show()

In [ ]:
# The full 87x87 matrix -- dense, but this is the complete evidence
# (the cell above is just a more readable subset of the same data).
fig, ax = plt.subplots(figsize=(26, 26))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
    ax=ax, xticks_rotation=90, colorbar=True, cmap='Purples', values_format='d'
)
ax.set_title(f'GemVision CNN — Full Confusion Matrix, all 87 classes (test accuracy {accuracy:.1%})')
fig.tight_layout()
fig.savefig('cnn_confusion_matrix_full.png', dpi=150)
plt.show()

## Save the evidence
Downloads the classification report and both confusion matrix images — these three files (plus a screenshot of the printed accuracy/report above) are the evidence to send your supervisor.

In [ ]:
with open('cnn_classification_report.txt', 'w') as f:
    f.write(f'Overall accuracy: {accuracy:.4f}\n\n')
    f.write(report)

files.download('cnn_classification_report.txt')
files.download('cnn_confusion_matrix_top20.png')
files.download('cnn_confusion_matrix_full.png')